In [ ]:
# Import necessary libraries
import pandas as pd
from neo4j import GraphDatabase
import matplotlib.pyplot as plt
import seaborn as sns

In [1]:
import pandas as pd

In [2]:
raw_directory = '../data/raw'

In [5]:
organization = pd.read_excel(raw_directory + "/organization" + ".xlsx")
project = pd.read_excel(raw_directory + "/project" + ".xlsx")

/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/.venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/.venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [4]:
organization.columns

Index(['projectID', 'projectAcronym', 'organisationID', 'vatNumber', 'name',
       'shortName', 'SME', 'activityType', 'street', 'postCode', 'city',
       'country', 'nutsCode', 'geolocation', 'organizationURL', 'contactForm',
       'contentUpdateDate', 'rcn', 'order', 'role', 'ecContribution',
       'netEcContribution', 'totalCost', 'endOfParticipation', 'active'],
      dtype='object')

In [10]:
# Create a DataFrame of unique organizations with ID, name, and country
unique_organizations = organization.drop_duplicates('organisationID')[['organisationID', 'name', 'country']]
unique_organizations.columns = ['organisationID', 'organisationName', 'country']
unique_organizations.reset_index(drop=True, inplace=True)
unique_organizations

,organisationID,organisationName,country
0,999981634,WAGENINGEN UNIVERSITY,NL
1,999997736,AARHUS UNIVERSITET,DK
2,999854855,UNIVERSITAET POTSDAM,DE
3,999990267,MAX-PLANCK-GESELLSCHAFT ZUR FORDERUNG DER WISS...,DE
4,999874546,UNIVERSIDAD COMPLUTENSE DE MADRID,ES
...,...,...,...
27262,885025576,ICOMPLAI UG,DE
27263,958284438,GREATER MANCHESTER POLICE,UK
27264,908208188,DPT - DEUTSCHER PRAVENTIONSTAG,DE
27265,951909016,LANDESKRIMINALAMT NIEDERSACHSEN,DE


In [11]:
from itertools import combinations
from collections import Counter

# Filter for 2024 projects only using startDate in project dataframe
project_2024_ids = project[project['startDate'].str.startswith('2024')]['id']

# Filter organization dataframe for those projects
organization_2024 = organization[organization['projectID'].isin(project_2024_ids)]

# Group by projectID and get all organizations per project
pairs = []
for project_id, group in organization_2024.groupby('projectID'):
    orgs = group['organisationID'].unique()
    pairs.extend(combinations(sorted(orgs), 2))

# Count the number of projects each pair worked on together
pair_counts = Counter(pairs)

# Prepare the simplified output DataFrame with only 3 columns
output_data = [
    (org1, org2, count) for (org1, org2), count in pair_counts.items()
]

output = pd.DataFrame(output_data, columns=[
    'organisation1_id', 'organisation2_id', 'num_of_projects'
])

output

,organisation1_id,organisation2_id,num_of_projects
0,999903355,999985029,1
1,999986096,999994826,2
2,974118330,998675238,1
3,974118330,999984059,1
4,998675238,999984059,1
...,...,...,...
257447,999587135,999990946,1
257448,999746409,999977366,1
257449,999746409,999977463,1
257450,999746409,999990946,1


In [38]:
# Import the Neo4j Python driver
from neo4j import GraphDatabase

# Neo4j connection parameters
URI = "bolt://localhost:7687"
USERNAME = "neo4j"  # Default username, change if different
PASSWORD = "password"  # Change to your actual password

# Function to create constraint (separate from data operations)
def create_constraint(session):
    try:
        session.run("CREATE CONSTRAINT org_id_constraint IF NOT EXISTS FOR (o:Organization) REQUIRE o.id IS UNIQUE")
        print("✅ Constraint created successfully")
    except Exception as e:
        print(f"Constraint creation error: {e}")

# Function to create organization nodes
def create_organization_nodes(tx, organizations_df):
    # Batch organizations into groups of 100 for more efficient processing
    batch_size = 100
    total_organizations = len(organizations_df)
    
    for i in range(0, total_organizations, batch_size):
        batch = organizations_df.iloc[i:i+batch_size]
        # Convert batch to list of dictionaries for cypher
        org_list = batch.to_dict('records')
        
        # Create organization nodes
        query = """
        UNWIND $organizations AS org
        MERGE (o:Organization {id: org.organisationID})
        SET o.name = org.organisationName,
            o.country = org.country
        """
        
        tx.run(query, organizations=org_list)
        print(f"Created/updated nodes {i} to {min(i+batch_size, total_organizations)} of {total_organizations}")

# Function to create relationships between organizations
def create_organization_relationships(tx, relationships_df):
    # Batch relationships into groups of 1000 for more efficient processing
    batch_size = 1000
    total_relationships = len(relationships_df)
    
    for i in range(0, total_relationships, batch_size):
        batch = relationships_df.iloc[i:i+batch_size]
        # Convert batch to list of dictionaries for cypher
        rel_list = batch.to_dict('records')
        
        # Create relationships
        query = """
        UNWIND $relationships AS rel
        MATCH (org1:Organization {id: rel.organisation1_id})
        MATCH (org2:Organization {id: rel.organisation2_id})
        MERGE (org1)-[r:COLLABORATES_WITH]-(org2)
        SET r.projects = rel.num_of_projects
        """
        
        tx.run(query, relationships=rel_list)
        print(f"Created/updated relationships {i} to {min(i+batch_size, total_relationships)} of {total_relationships}")

# Convert IDs to strings to avoid type issues
def convert_ids_to_strings(unique_orgs_df, output_df):
    print("Converting organization IDs to strings...")
    # Make copies to avoid modifying the originals
    unique_orgs_copy = unique_orgs_df.copy()
    output_copy = output_df.copy()
    
    # Convert IDs to strings
    unique_orgs_copy['organisationID'] = unique_orgs_copy['organisationID'].astype(str)
    output_copy['organisation1_id'] = output_copy['organisation1_id'].astype(str)
    output_copy['organisation2_id'] = output_copy['organisation2_id'].astype(str)
    
    return unique_orgs_copy, output_copy

# Main function to populate the Neo4j database
def populate_neo4j():
    # Convert IDs to strings to avoid type issues
    orgs_df, rels_df = convert_ids_to_strings(unique_organizations, output)
    
    # Connect to Neo4j
    try:
        with GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD)) as driver:
            # First, create constraint in a separate session
            print("Creating constraint...")
            with driver.session() as session:
                create_constraint(session)
            
            # Then create organization nodes
            print("\nCreating organization nodes...")
            with driver.session() as session:
                session.execute_write(create_organization_nodes, orgs_df)
            
            # Finally create relationships
            print("\nCreating organization relationships...")
            with driver.session() as session:
                session.execute_write(create_organization_relationships, rels_df)
                
            print("\nDone! Neo4j database has been populated with:")
            print(f"- {len(unique_organizations)} organization nodes")
            print(f"- {len(output)} collaboration relationships")
    except Exception as e:
        print(f"Error connecting to Neo4j: {e}")
        print("Please check that your Neo4j database is running and that the credentials are correct.")

# Execute the main function
populate_neo4j()

Converting organization IDs to strings...
Creating constraint...
✅ Constraint created successfully

Creating organization nodes...
Created/updated nodes 0 to 100 of 27267
Created/updated nodes 100 to 200 of 27267
Created/updated nodes 200 to 300 of 27267
Created/updated nodes 300 to 400 of 27267
Created/updated nodes 400 to 500 of 27267
Created/updated nodes 500 to 600 of 27267
Created/updated nodes 600 to 700 of 27267
Created/updated nodes 700 to 800 of 27267
Created/updated nodes 800 to 900 of 27267
Created/updated nodes 900 to 1000 of 27267
Created/updated nodes 1000 to 1100 of 27267
Created/updated nodes 1100 to 1200 of 27267
Created/updated nodes 1200 to 1300 of 27267
Created/updated nodes 1300 to 1400 of 27267
Created/updated nodes 1400 to 1500 of 27267
Created/updated nodes 1500 to 1600 of 27267
Created/updated nodes 1600 to 1700 of 27267
Created/updated nodes 1700 to 1800 of 27267
Created/updated nodes 1800 to 1900 of 27267
Created/updated nodes 1900 to 2000 of 27267
Created/up

In [41]:
with driver.session() as session:
    session.run("""
        MATCH (a:Organization)-[r:COLLABORATES_WITH]->(b:Organization)
        MERGE (b)-[r2:COLLABORATES_WITH]->(a)
        SET r2.projects = r.projects
    """)

/var/folders/1g/drrbdl5d5tz58v9sn33gqfb40000gn/T/ipykernel_94043/3507555807.py:1: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:


In [42]:
# Comprehensive centrality analysis using Neo4j Graph Data Science
def run_centrality_analysis():
    # Neo4j connection parameters
    URI = "bolt://localhost:7687"
    USERNAME = "neo4j"
    PASSWORD = "password"
    
    # Connect to Neo4j
    with GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD)) as driver:
        with driver.session() as session:
            # Create a named graph projection for centrality algorithms
            print("Creating graph projection...")
            create_graph = """
            CALL gds.graph.project(
                'orgCollabGraph',
                'Organization', 
                'COLLABORATES_WITH',
                {
                    relationshipProperties: ['projects']
                }
            )
            YIELD graphName, nodeCount, relationshipCount
            RETURN graphName, nodeCount, relationshipCount
            """
            
            # Try to create the graph, drop if it exists already
            try:
                result = session.run(create_graph)
                graph_stats = result.single()
                print(f"Created graph with {graph_stats['nodeCount']} nodes and {graph_stats['relationshipCount']} relationships")
            except Exception:
                # Graph might already exist, drop it and recreate
                session.run("CALL gds.graph.drop('orgCollabGraph', false) YIELD graphName")
                result = session.run(create_graph)
                graph_stats = result.single()
                print(f"Recreated graph with {graph_stats['nodeCount']} nodes and {graph_stats['relationshipCount']} relationships")
            
            # 1. Run PageRank
            print("Running PageRank algorithm...")
            pagerank_query = """
            CALL gds.pageRank.stream('orgCollabGraph', {
                maxIterations: 20,
                relationshipWeightProperty: 'projects'
            })
            YIELD nodeId, score
            RETURN nodeId, score as pagerank_score
            """
            
            result = session.run(pagerank_query)
            pagerank_records = {record["nodeId"]: record["pagerank_score"] for record in result}
            
            # 2. Run weighted degree centrality
            print("Running weighted degree centrality...")
            degree_query = """
            CALL gds.degree.stream('orgCollabGraph', {
                relationshipWeightProperty: 'projects'
            })
            YIELD nodeId, score
            RETURN nodeId, score as degree_score
            """
            
            result = session.run(degree_query)
            degree_records = {record["nodeId"]: record["degree_score"] for record in result}
            
            # 3. Run weighted betweenness centrality
            print("Running betweenness centrality (this may take a while)...")
            betweenness_query = """
            CALL gds.betweenness.stream('orgCollabGraph', {
                relationshipWeightProperty: 'projects'
            })
            YIELD nodeId, score
            RETURN nodeId, score as betweenness_score
            """
            
            result = session.run(betweenness_query)
            betweenness_records = {record["nodeId"]: record["betweenness_score"] for record in result}
            
            # Get organization names and combine all scores
            print("Combining results...")
            org_name_query = """
            MATCH (o:Organization)
            RETURN id(o) as nodeId, o.name as organization_name
            """
            
            result = session.run(org_name_query)
            
            # Create comprehensive DataFrame
            centrality_data = []
            for record in result:
                node_id = record["nodeId"]
                centrality_data.append({
                    "organization_name": record["organization_name"],
                    "pagerank_score": pagerank_records.get(node_id, 0),
                    "degree_centrality_score": degree_records.get(node_id, 0),
                    "betweenness_score": betweenness_records.get(node_id, 0)
                })
            
            # Clean up by dropping the graph
            session.run("CALL gds.graph.drop('orgCollabGraph', false)")
            
            # Create DataFrame with all centrality measures
            centrality_df = pd.DataFrame(centrality_data)
            
            # Sort by PageRank score (default ranking)
            centrality_df = centrality_df.sort_values(by='pagerank_score', ascending=False).reset_index(drop=True)
            
            # Show top 10 organizations by PageRank
            print(f"\nTop 10 organizations by PageRank (from {len(centrality_df)} total):")
            print(centrality_df[['organization_name', 'pagerank_score', 'degree_centrality_score', 'betweenness_score']].head(10))
            
            return centrality_df

# Run the centrality analysis
centrality_results = run_centrality_analysis()

Creating graph projection...
Created graph with 27267 nodes and 514904 relationships
Running PageRank algorithm...
Running weighted degree centrality...
Running betweenness centrality (this may take a while)...
Combining results...


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated function: `id`.} {position: line: 3, column: 20, offset: 55} for query: '\n            MATCH (o:Organization)\n            RETURN id(o) as nodeId, o.name as organization_name\n            '
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated field from a procedure. ('schema' returned by 'gds.graph.drop' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL gds.graph.drop('orgCollabGraph', false)"



Top 10 organizations by PageRank (from 27267 total):
                                   organization_name  pagerank_score  \
0  FRAUNHOFER GESELLSCHAFT ZUR FORDERUNG DER ANGE...       58.978647   
1  CENTRE NATIONAL DE LA RECHERCHE SCIENTIFIQUE CNRS       53.871328   
2  AGENCIA ESTATAL CONSEJO SUPERIOR DE INVESTIGAC...       52.731592   
3                 CONSIGLIO NAZIONALE DELLE RICERCHE       40.552454   
4                     KATHOLIEKE UNIVERSITEIT LEUVEN       37.045673   
5  COMMISSARIAT A L ENERGIE ATOMIQUE ET AUX ENERG...       32.710827   
6                      TECHNISCHE UNIVERSITEIT DELFT       30.966898   
7                              POLITECNICO DI MILANO       30.748638   
8                  TEKNOLOGIAN TUTKIMUSKESKUS VTT OY       29.686142   
9                      DANMARKS TEKNISKE UNIVERSITET       29.623270   

   degree_centrality_score  betweenness_score  
0                   3151.0       9.143934e+06  
1                   3272.0       3.426547e+06  
2        

In [44]:
centrality_results.to_csv('../data/processed/centrality_analysis.csv', index=False)

In [49]:
# Add country information to centrality results
URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "password"

with GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD)) as driver:
    with driver.session() as session:
        # Get organization names and countries
        org_country_query = """
        MATCH (o:Organization)
        RETURN o.name as organization_name, o.country as country
        """
        result = session.run(org_country_query)
        org_country_dict = {record["organization_name"]: record["country"] for record in result}

# Add country column to centrality_results
centrality_results['country'] = centrality_results['organization_name'].map(org_country_dict)

# Display top organizations with country information
print("Top 10 organizations by PageRank with country information:")
print(centrality_results[['organization_name', 'country', 'pagerank_score', 'degree_centrality_score', 'betweenness_score']].head(10))

# Save updated DataFrame with country information
centrality_results.to_csv('../data/processed/centrality_analysis.csv', index=False)

Top 10 organizations by PageRank with country information:
                                   organization_name country  pagerank_score  \
0  FRAUNHOFER GESELLSCHAFT ZUR FORDERUNG DER ANGE...      DE       58.978647   
1  CENTRE NATIONAL DE LA RECHERCHE SCIENTIFIQUE CNRS      FR       53.871328   
2  AGENCIA ESTATAL CONSEJO SUPERIOR DE INVESTIGAC...      ES       52.731592   
3                 CONSIGLIO NAZIONALE DELLE RICERCHE      IT       40.552454   
4                     KATHOLIEKE UNIVERSITEIT LEUVEN      BE       37.045673   
5  COMMISSARIAT A L ENERGIE ATOMIQUE ET AUX ENERG...      FR       32.710827   
6                      TECHNISCHE UNIVERSITEIT DELFT      NL       30.966898   
7                              POLITECNICO DI MILANO      IT       30.748638   
8                  TEKNOLOGIAN TUTKIMUSKESKUS VTT OY      FI       29.686142   
9                      DANMARKS TEKNISKE UNIVERSITET      DK       29.623270   

   degree_centrality_score  betweenness_score  
0           

In [ ]:
# Analyze country distribution in communities and centrality scores
import matplotlib.pyplot as plt
import seaborn as sns

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# 1. Top countries by average PageRank
country_pagerank = centrality_results.groupby('country')['pagerank_score'].agg(['mean', 'count'])
country_pagerank = country_pagerank[country_pagerank['count'] >= 5]  # Filter for countries with at least 5 orgs
country_pagerank = country_pagerank.sort_values('mean', ascending=False).head(10)

# Plot average PageRank by country
country_pagerank['mean'].plot(kind='bar', ax=ax1, color='skyblue')
ax1.set_title('Top 10 Countries by Average PageRank', fontsize=14)
ax1.set_ylabel('Average PageRank Score')
ax1.set_xlabel('Country')

# 2. Country representation in top communities
top_communities = louvain_df['communityId'].value_counts().head(5).index.tolist()
community_country_data = []

for comm_id in top_communities:
    comm_df = louvain_df[louvain_df['communityId'] == comm_id]
    country_counts = comm_df['country'].value_counts().head(3)
    for country, count in country_counts.items():
        community_country_data.append({
            'Community': f'Comm {comm_id}',
            'Country': country,
            'Count': count
        })

community_country_df = pd.DataFrame(community_country_data)

# Create grouped bar chart
sns.barplot(x='Community', y='Count', hue='Country', data=community_country_df, ax=ax2)
ax2.set_title('Top 3 Countries in 5 Largest Communities', fontsize=14)
ax2.set_ylabel('Number of Organizations')
ax2.legend(title='Country', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [47]:
# Run Louvain community detection using Neo4j GDS (compatible with older GDS versions)
URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "password"
with GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD)) as driver:
    with driver.session() as session:
        # Ensure the graph projection exists
        print("Creating graph projection for Louvain...")
        create_graph = """
        CALL gds.graph.project(
            'orgCollabGraph',
            'Organization',
            'COLLABORATES_WITH',
            {
                relationshipProperties: ['projects']
            }
        )
        YIELD graphName, nodeCount, relationshipCount
        RETURN graphName, nodeCount, relationshipCount
        """
        try:
            result = session.run(create_graph)
            graph_stats = result.single()
            print(f"Created graph with {graph_stats['nodeCount']} nodes and {graph_stats['relationshipCount']} relationships")
        except Exception:
            session.run("CALL gds.graph.drop('orgCollabGraph', false) YIELD graphName")
            result = session.run(create_graph)
            graph_stats = result.single()
            print(f"Recreated graph with {graph_stats['nodeCount']} nodes and {graph_stats['relationshipCount']} relationships")

        # Run Louvain algorithm (no score output)
        print("Running Louvain community detection...")
        louvain_query = """
        CALL gds.louvain.stream('orgCollabGraph', {
            relationshipWeightProperty: 'projects'
        })
        YIELD nodeId, communityId
        MATCH (o:Organization) WHERE id(o) = nodeId
        RETURN o.name AS organization_name, communityId
        ORDER BY communityId
        """
        result = session.run(louvain_query)
        louvain_records = list(result)
        session.run("CALL gds.graph.drop('orgCollabGraph', false)")

        # Create DataFrame
        louvain_df = pd.DataFrame([
            {"organization_name": r["organization_name"], "communityId": r["communityId"]}
            for r in louvain_records
        ])
        print(f"\nTotal nodes: {len(louvain_df)}")
        n_communities = louvain_df['communityId'].nunique()
        print(f"Total communities: {n_communities}")

        # Community sizes
        comm_sizes = louvain_df['communityId'].value_counts()
        print(f"Largest community size: {comm_sizes.iloc[0]}")
        print(f"Number of singleton communities (size=1): {(comm_sizes==1).sum()}")

        # Show top 5 largest communities
        print("\nTop 5 largest communities:")
        for cid, size in comm_sizes.head(5).items():
            members = louvain_df[louvain_df['communityId']==cid]['organization_name'].tolist()
            print(f"Community {cid}: size={size}, members={members[:5]}{'...' if size>5 else ''}")

        # Save to CSV
        louvain_df.to_csv('../data/processed/organization_communities_louvain.csv', index=False)
        print("\nLouvain community assignments exported to: ../data/processed/organization_communities_louvain.csv")

Creating graph projection for Louvain...
Created graph with 27267 nodes and 514904 relationships
Running Louvain community detection...
Created graph with 27267 nodes and 514904 relationships
Running Louvain community detection...


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated function: `id`.} {position: line: 6, column: 38, offset: 186} for query: "\n        CALL gds.louvain.stream('orgCollabGraph', {\n            relationshipWeightProperty: 'projects'\n        })\n        YIELD nodeId, communityId\n        MATCH (o:Organization) WHERE id(o) = nodeId\n        RETURN o.name AS organization_name, communityId\n        ORDER BY communityId\n        "
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated field from a procedure. ('schema' returned by 'gds.graph.drop' is deprecated.)} {pos


Total nodes: 27267
Total communities: 14936
Largest community size: 1824
Number of singleton communities (size=1): 14898

Top 5 largest communities:
Community 20653: size=1824, members=['CENTRE NATIONAL DE LA RECHERCHE SCIENTIFIQUE CNRS', 'EUROPEAN GRAVITATIONAL OBSERVATORY(EGO) (OSSERVATORIO GRAVITAZIO NALEEUROPEO)', 'ASTOS SOLUTIONS SRL', 'TYVAK INTERNATIONAL SRL', 'ROYAL BOTANIC GARDENS KEW']...
Community 12111: size=1549, members=['GENEGIS GI SRL', 'ELVALHALCOR ELLINIKI VIOMIHANIA HALKOU KAI ALOUMINIOU ANONYMOS ETAIREIA', 'EREVNITIKO PANEPISTIMIAKO INSTITOUTO SYSTIMATON EPIKOINONION KAI YPOLOGISTON', 'ASOCIACION DE EMPRESAS TECNOLOGICAS INNOVALIA', 'ATLANTIS ENGINEERING AE']...
Community 1773: size=1198, members=['BIOSENSE INSTITUTE - RESEARCH AND DEVELOPMENT INSTITUTE FOR INFORMATION TECHNOLOGIES IN BIOSYSTEMS', 'SUTAS SUT URUNLERI AS', 'TEAMIT RESEARCH SL', 'CONSORCIO PARA LA EXPLOTACION DEL CENTRO NACIONAL DE ANALISIS GENOMICO', 'INSTITUT ZA MOLEKULARNU GENETIKU I GENETICKO INZ

In [ ]:
# Add country information to Louvain community detection results
URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "password"

with GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD)) as driver:
    with driver.session() as session:
        # Get organization names and countries
        org_country_query = """
        MATCH (o:Organization)
        RETURN o.name as organization_name, o.country as country
        """
        result = session.run(org_country_query)
        org_country_dict = {record["organization_name"]: record["country"] for record in result}

# Add country column to louvain_df
louvain_df['country'] = louvain_df['organization_name'].map(org_country_dict)

# Print community information with country data
n_communities = louvain_df['communityId'].nunique()
print(f"\nTotal communities: {n_communities}")

# Count organizations by country in each community
print("\nTop 5 communities with country distribution:")
for cid in louvain_df['communityId'].value_counts().head(5).index:
    community = louvain_df[louvain_df['communityId']==cid]
    country_counts = community['country'].value_counts()
    size = len(community)
    print(f"\nCommunity {cid} (size={size}):")
    print(f"Country distribution: {dict(country_counts.head(5))}")
    print(f"Sample members: {community['organization_name'].head(3).tolist()}...")

# Save updated DataFrame with country information
louvain_df.to_csv('../data/processed/organization_communities_louvain.csv', index=False)